# Universal Analyst - Lightning.ai one-click runner

This notebook prepares the cloud runtime without touching your local machine.

It will:
1. Install backend Python dependencies when missing.
2. Install Node.js 20 when missing.
3. Install and start Ollama, then pull the required Gemma 4 models.
4. Start the FastAPI backend on :8000.
5. Build/serve the React frontend on :5173.


## 1. Detect repo root + define paths

In [ ]:
import os, sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    """Walk up from `start` looking for a dir that has both backend/ and frontend/."""
    for candidate in [start, *start.parents]:
        if (candidate / 'backend').is_dir() and (candidate / 'frontend').is_dir():
            return candidate
    return start

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
BACKEND_DIR = REPO_ROOT / 'backend'
FRONTEND_DIR = REPO_ROOT / 'frontend'
REQUIREMENTS = BACKEND_DIR / 'requirements.txt'

assert BACKEND_DIR.is_dir(), f'backend/ not found under {REPO_ROOT}'
assert FRONTEND_DIR.is_dir(), f'frontend/ not found under {REPO_ROOT}'

print(f'[ok] REPO_ROOT    = {REPO_ROOT}')
print(f'[ok] BACKEND_DIR  = {BACKEND_DIR}')
print(f'[ok] FRONTEND_DIR = {FRONTEND_DIR}')
print(f'[ok] Python       = {sys.version.split()[0]} ({sys.executable})')

## 2. Install Python dependencies

Single source of truth: `backend/requirements.txt`. The cell first checks whether each dependency imports cleanly; it only runs `pip install` when something is missing.

In [ ]:
import importlib.util
import re
import subprocess
import sys

PACKAGE_IMPORTS = {
    'pydantic-settings': 'pydantic_settings',
    'python-multipart': 'multipart',
    'sentence-transformers': 'sentence_transformers',
    'scikit-learn': 'sklearn',
    'pytest-cov': 'pytest_cov',
}

REQ_NAME_RE = re.compile(r'([A-Za-z0-9_.-]+)')

def _requirement_name(line: str) -> str | None:
    line = line.split('#', 1)[0].strip()
    if not line or line.startswith(('-', 'git+', 'http://', 'https://')):
        return None
    match = REQ_NAME_RE.match(line)
    return match.group(1).lower() if match else None

def _import_name(package_name: str) -> str:
    return PACKAGE_IMPORTS.get(package_name, package_name.replace('-', '_'))

def _missing_requirements(req_path):
    missing = []
    for line in req_path.read_text(encoding='utf-8').splitlines():
        package = _requirement_name(line)
        if not package:
            continue
        module = _import_name(package)
        if importlib.util.find_spec(module) is None:
            missing.append(package)
    return sorted(set(missing))

missing = _missing_requirements(REQUIREMENTS)
if missing:
    print('Missing Python dependencies: ' + ', '.join(missing))
    print(f'Installing from {REQUIREMENTS} ...')
    ret = subprocess.call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQUIREMENTS)])
    if ret == 0:
        print('Python dependencies installed.')
    else:
        print(f'pip install returned {ret}; review output above. Continuing.')
else:
    print('Python dependencies already import cleanly; skipping pip install.')

## 3. Install Node.js 20 (if missing)

In [ ]:
import shutil, subprocess

def _node_ok() -> bool:
    if shutil.which('node') is None:
        return False
    try:
        subprocess.check_output(['node', '-v'])
        return True
    except Exception:
        return False

if _node_ok():
    v = subprocess.check_output(['node', '-v']).decode().strip()
    print(f'[ok] Node.js already installed: {v}')
else:
    print('[wait] Node.js not found - installing Node 20 via NodeSource (Debian/Ubuntu)...')
    install = (
        'curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && '
        'apt-get install -y nodejs'
    )
    # Use sudo if not root.
    if os.geteuid() != 0:
        install = 'sudo bash -c ' + repr(install)
    else:
        install = 'bash -c ' + repr(install)
    ret = subprocess.call(install, shell=True)
    if ret == 0 and _node_ok():
        print('[ok] Node.js installed.')
    else:
        print(f'[warn] Node install returned {ret}. The frontend build cell may fail.')

for tool in ('node', 'npm'):
    if shutil.which(tool):
        try:
            print(f'   {tool}: ' + subprocess.check_output([tool, '-v']).decode().strip())
        except Exception:
            pass

## 4. Install + start Ollama

In [ ]:
import shutil, subprocess, time, httpx

OLLAMA_URL = 'http://localhost:11434'

# Install Ollama if absent.
if shutil.which('ollama') is None:
    print('[wait] Installing Ollama...')
    ret = subprocess.call('curl -fsSL https://ollama.com/install.sh | sh', shell=True)
    print('[ok] Ollama installed.' if ret == 0 else f'[warn] Ollama install returned {ret}.')
else:
    print('[ok] Ollama already installed.')

def _ollama_ready() -> bool:
    try:
        return httpx.get(f'{OLLAMA_URL}/api/tags', timeout=2.0).status_code == 200
    except Exception:
        return False

# Start the server in the background if it is not already responding.
if _ollama_ready():
    print('[ok] Ollama server already running.')
else:
    print('[wait] Starting `ollama serve` (log: /tmp/ollama.log)...')
    subprocess.Popen('nohup ollama serve > /tmp/ollama.log 2>&1 &', shell=True)
    for i in range(30):
        if _ollama_ready():
            break
        time.sleep(1)

if _ollama_ready():
    print('[ok] Ollama server is ready at ' + OLLAMA_URL)
else:
    print('[error] Ollama server did not become ready. See /tmp/ollama.log')

## 5. Pull required models

- **MAIN**: gemma4:12b
- **ROUTER**: gemma4:e4b

The cell below pulls the exact model tags configured by the backend. If a tag is unavailable, the setup stops with a real error so the runtime can be fixed directly.

In [ ]:
import subprocess

def pull_required_model(label, tag):
    """Pull one required Ollama tag and raise if it fails."""
    print(f'[model] [{label}] pulling {tag} ...')
    ret = subprocess.call(['ollama', 'pull', tag])
    if ret != 0:
        raise RuntimeError(f'[{label}] required model {tag} failed to pull (rc={ret}).')
    print(f'[model] [{label}] ready: {tag}')
    return tag

MAIN_MODEL = pull_required_model('MAIN', 'gemma4:12b')
ROUTER_MODEL = pull_required_model('ROUTER', 'gemma4:e4b')

print()
print(f'MAIN_MODEL   = {MAIN_MODEL}')
print(f'ROUTER_MODEL = {ROUTER_MODEL}')

## 6. GPU check (non-fatal)

In [ ]:
import subprocess

try:
    out = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
        stderr=subprocess.STDOUT,
    ).decode().strip()
    print('[ok] GPU detected:')
    print(out)
except Exception as exc:
    print(f'[warn] nvidia-smi unavailable ({exc}). Continuing - Ollama will use CPU if no GPU.')

## 7. Start the FastAPI backend (:8000)

In [ ]:
import os, sys, subprocess, time, httpx

backend_env = os.environ.copy()
backend_env.update({
    'OLLAMA_HOST': 'http://localhost:11434',
    'MODEL_NAME': MAIN_MODEL,
    'ROUTER_MODEL_NAME': ROUTER_MODEL,
    'CHROMA_PERSIST_DIRECTORY': './chroma_data',
    'ENVIRONMENT': 'production',
    'ALLOWED_ORIGINS': '*',
    'IMAGE_CHAT_ENABLED': 'true',
    'MAX_CHAT_IMAGES': '3',
    'MAX_CHAT_IMAGE_MB': '5',
    'CONTEXT_WINDOW_TOKENS': '256000',
})

def _backend_ready() -> bool:
    try:
        return httpx.get('http://localhost:8000/health', timeout=2.0).status_code == 200
    except Exception:
        return False

if _backend_ready():
    print('[ok] Backend already running on :8000.')
    backend_proc = None
else:
    print('[wait] Starting uvicorn (app.main:app) on 0.0.0.0:8000 ...')
    backend_proc = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
        cwd=str(BACKEND_DIR),
        env=backend_env,
    )
    for i in range(60):
        if _backend_ready():
            break
        time.sleep(1)

if _backend_ready():
    print('[ok] Backend healthy at http://localhost:8000  (docs: http://localhost:8000/docs)')
else:
    print('[error] Backend did not become healthy. Check the output above.')

## 8. Build the frontend + serve it with a `/api` proxy (:5173)

The build cell skips `npm install` when `node_modules` already contains every dependency from `frontend/package.json`, then always runs `npm run build` so the served SPA reflects the current source.

In [ ]:
import json
import subprocess

NODE_MODULES = FRONTEND_DIR / 'node_modules'
PACKAGE_JSON = FRONTEND_DIR / 'package.json'

def _node_package_present(package_name: str) -> bool:
    parts = package_name.split('/')
    return (NODE_MODULES.joinpath(*parts) / 'package.json').exists()

def _missing_node_dependencies():
    pkg = json.loads(PACKAGE_JSON.read_text(encoding='utf-8'))
    required = {}
    required.update(pkg.get('dependencies', {}))
    required.update(pkg.get('devDependencies', {}))
    return [name for name in sorted(required) if not _node_package_present(name)]

missing_node = _missing_node_dependencies()
if missing_node:
    print('Missing npm packages: ' + ', '.join(missing_node[:12]) + (' ...' if len(missing_node) > 12 else ''))
    print('npm install ...')
    ret = subprocess.call(['npm', 'install'], cwd=str(FRONTEND_DIR))
    print('npm install done.' if ret == 0 else f'npm install rc={ret}')
else:
    print('npm dependencies already present; skipping npm install.')

print('npm run build ...')
ret = subprocess.call(['npm', 'run', 'build'], cwd=str(FRONTEND_DIR))
if ret == 0 and (FRONTEND_DIR / 'dist' / 'index.html').exists():
    print('Frontend built -> frontend/dist')
else:
    print(f'Frontend build failed (rc={ret}). The serve cell will not work.')

In [ ]:
import threading
import urllib.request, urllib.error
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer

DIST_DIR = str(FRONTEND_DIR / 'dist')
BACKEND_BASE = 'http://localhost:8000'
FRONTEND_PORT = 5173

class SPAProxyHandler(SimpleHTTPRequestHandler):
    """Serve the SPA from dist/ and reverse-proxy /api/* to the backend.

    - /api/<path>  -> BACKEND_BASE/<path>  (strips the /api prefix)
    - everything else -> static files, with SPA history routing to index.html
    """

    def _proxy(self):
        # Strip leading '/api' -> backend path at ROOT.
        backend_path = self.path[len('/api'):] or '/'
        url = BACKEND_BASE + backend_path
        length = int(self.headers.get('Content-Length', 0) or 0)
        body = self.rfile.read(length) if length else None
        fwd_headers = {k: v for k, v in self.headers.items()
                       if k.lower() not in ('host', 'content-length', 'connection')}
        req = urllib.request.Request(url, data=body, method=self.command, headers=fwd_headers)
        try:
            with urllib.request.urlopen(req, timeout=300) as resp:
                self.send_response(resp.status)
                for k, v in resp.headers.items():
                    if k.lower() in ('transfer-encoding', 'connection', 'content-encoding'):
                        continue
                    self.send_header(k, v)
                self.end_headers()
                while True:
                    chunk = resp.read(8192)
                    if not chunk:
                        break
                    self.wfile.write(chunk)
                    self.wfile.flush()  # stream SSE through promptly
        except urllib.error.HTTPError as e:
            self.send_response(e.code)
            self.end_headers()
            self.wfile.write(e.read())
        except Exception as e:
            self.send_response(502)
            self.end_headers()
            self.wfile.write(str(e).encode())

    def do_GET(self):
        if self.path.startswith('/api/') or self.path == '/api':
            return self._proxy()
        return super().do_GET()

    def do_POST(self):
        return self._proxy()

    def do_PUT(self):
        return self._proxy()

    def do_DELETE(self):
        if self.path.startswith('/api/') or self.path == '/api':
            return self._proxy()
        self.send_response(405)
        self.end_headers()

    def send_head(self):
        # SPA history routing: serve index.html for non-file, non-/api routes.
        from pathlib import Path as _P
        rel = self.path.split('?', 1)[0].lstrip('/')
        target = _P(self.directory) / rel
        if rel and not target.exists():
            self.path = '/index.html'
        return super().send_head()

    def log_message(self, *args):
        pass  # quiet

def _serve():
    handler = partial(SPAProxyHandler, directory=DIST_DIR)
    httpd = ThreadingHTTPServer(('0.0.0.0', FRONTEND_PORT), handler)
    httpd.serve_forever()

# Avoid double-starting on re-run.
if not globals().get('_frontend_server_started'):
    t = threading.Thread(target=_serve, daemon=True)
    t.start()
    globals()['_frontend_server_started'] = True
    print(f'[ok] Serving SPA + /api proxy on http://0.0.0.0:{FRONTEND_PORT}')
else:
    print('[ok] Frontend proxy server already running.')

print('   Open the platform-forwarded port 5173 in your browser.')
print('   API docs: the forwarded port 8000 -> /docs')

## 9. Smoke tests (all non-fatal)

In [ ]:
import httpx

try:
    r = httpx.get('http://localhost:8000/health', timeout=5.0)
    print(f'[ok] /health -> {r.status_code} {r.json()}')
except Exception as exc:
    print(f'[error] /health failed: {exc}')

In [ ]:
import httpx

SAMPLE = REPO_ROOT / 'data' / 'samples' / 'sales_data.csv'
session_id = None
try:
    with open(SAMPLE, 'rb') as f:
        r = httpx.post(
            'http://localhost:8000/upload',
            files={'file': ('sales_data.csv', f, 'text/csv')},
            timeout=120.0,
        )
    data = r.json()
    session_id = data.get('session_id')
    profile = data.get('profile', {})
    eda = data.get('eda', {})
    print(f'[ok] /upload -> session_id={session_id}')
    print(f'   profile: rows={profile.get("row_count")} cols={profile.get("column_count")}'
          f' file={profile.get("file_name")}')
    print(f'   Static visuals: {len(eda.get("charts", []))} (static visuals disabled); suggestions: {len(eda.get("suggestions", []))}')
except Exception as exc:
    print(f'[error] /upload failed: {exc}')

In [ ]:
import httpx

# Stream a bit of the SSE /chat response to confirm skill/token events.
if session_id:
    try:
        seen_events = []
        with httpx.stream(
            'POST', 'http://localhost:8000/chat',
            json={'session_id': session_id, 'message': 'What are the top selling products?'},
            timeout=180.0,
        ) as resp:
            print(f'   /chat status: {resp.status_code}')
            token_preview = ''
            for line in resp.iter_lines():
                if line.startswith('event:'):
                    seen_events.append(line.split(':', 1)[1].strip())
                elif line.startswith('data:') and 'token' in line:
                    token_preview += line
                # Stop once we have proof of life.
                if len(seen_events) >= 4 and len(token_preview) > 80:
                    break
        print(f'[ok] /chat events seen: {seen_events[:8]}')
    except Exception as exc:
        print(f'[warn] /chat smoke check issue: {exc}')
else:
    print('[warn] Skipping /chat - no session_id from upload.')

In [ ]:
import httpx, json

if session_id:
    try:
        r = httpx.post(
            'http://localhost:8000/visualize',
            json={
                'session_id': session_id,
                'chart_spec': {'type': 'bar', 'x': 'product', 'y': 'sales',
                               'title': 'Sales by Product'},
            },
            timeout=60.0,
        )
        if r.status_code == 200:
            chart = json.loads(r.json()['chart'])
            n_traces = len(chart.get('data', []))
            print(f'[ok] /visualize -> {r.status_code}; chart has {n_traces} trace(s).')
        else:
            print(f'[warn] /visualize -> {r.status_code}: {r.text[:200]}')
    except Exception as exc:
        print(f'[warn] /visualize smoke check issue: {exc}')
else:
    print('[warn] Skipping /visualize - no session_id from upload.')

### Troubleshooting
- **Models:** if a required gemma4:* tag is unavailable, the pull cell raises a real error. Pull or retag the required model in Ollama, then rerun the cell.
- **Ports:** open the forwarded :5173 URL for the app and :8000/docs for the backend API.
- **Logs:** backend logs are written in the notebook output and the frontend server logs appear in the final serve cell.
